# Copia corregida — requiere ejecutar todas las celdas
Original preservado. Cambios: normalizacion coherente por modelo interno, metricas no estimables como N/D, retiro de cifras historicas y de la resta de perdidas no comparables; exportacion de predicciones para auditoria.
No se ha ejecutado el entrenamiento de esta copia. Usar en Google Colab con SkeletonData.zip; descargar un ZIP nuevo al terminar. Guardar tambien el notebook con sus salidas. No publicar automaticamente identificadores individuales del dataset.


1. Subir este archivo .ipynb a Colab y conectar un entorno de ejecución Python. El código usa CPU; no necesita GPU.
2. Ejecutar las celdas en orden. La celda de carga solicitará **SkeletonData.zip** (aproximadamente 199 MB), el dataset público.



In [ ]:
import importlib.util
import subprocess
import sys
paquetes = {"numpy": "numpy", "sklearn": "scikit-learn", "torch": "torch", "matplotlib": "matplotlib"}
faltantes = [paquete for modulo, paquete in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
print("Dependencias disponibles. Python:", sys.version)


In [ ]:
from google.colab import files
from pathlib import Path
import os
import hashlib
import zipfile

carpeta_tp = Path("/content/irds_tp")
carpeta_tp.mkdir(parents=True, exist_ok=True)
os.chdir(carpeta_tp)
dataset = carpeta_tp / "SkeletonData.zip"
if not dataset.exists():
    print("Selecciona SkeletonData.zip en tu computador y espera a que termine la carga.")
    cargados = files.upload()
    del cargados
if not dataset.exists():
    raise FileNotFoundError("Falta SkeletonData.zip. Vuelve a ejecutar esta celda y selecciona ese archivo.")
md5 = hashlib.md5()
with dataset.open("rb") as archivo:
    for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
        md5.update(bloque)
if md5.hexdigest() != "c243bcdbd1492e4928032e158accdd29":
    raise ValueError("El ZIP no coincide con el dataset auditado. Verifica la descarga antes de entrenar.")
with zipfile.ZipFile(dataset) as archivo:
    if not any("/Simplified/" in nombre and nombre.endswith(".txt") for nombre in archivo.namelist()):
        raise ValueError("No se encontró la estructura Simplified esperada.")
print("Dataset verificado. Directorio de trabajo:", carpeta_tp)


# TP de Redes Neuronales: clasificación de movimiento con una CNN temporal


## Autores

Laura Daniela Muñoz Torres

Anna Carla

Carlos Catán

Jonathan Caceres Prada

## Qué se investiga

Clasificación de la etiqueta publicada correcta/incorrecta de un único gesto, con evaluación por participante y comparación contra regresión logística. No es validación clínica.


# TP Redes Neuronales — Primer experimento sobre IntelliRehabDS (gesto 2 del dataset)
**Pregunta:** ¿qué desempeño tiene una CNN temporal pequeña para reproducir la etiqueta publicada
correcta/incorrecta del gesto 2, frente a una regresión logística sobre características de movimiento,
evaluando siempre en participantes que el modelo no vio al entrenar?

**Unidad de análisis:** una secuencia completa (una repetición). **Agrupación:** participante.
Diseño fijado ANTES de entrenar en `criterios_v1.md`. Nada aquí es validación clínica.
Fuente: IntelliRehabDS v2.0.1 (Miron et al., CC BY 4.0), https://zenodo.org/records/4610859

In [ ]:
import json, zipfile, io, pathlib, platform, sys, collections, hashlib, time
import numpy as np, sklearn, torch, torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    log_loss, matthews_corrcoef, roc_auc_score,
)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
# Fuente canonica: SkeletonData.zip (MD5 c243bcdb...). Si no esta, se usa el reempaquetado de la carpeta
# Simplified (simplified_partial.zip), cuyo contenido se verifico contra audit_manifest.json (2.589/2.589).
ZIP = next(p for p in [ROOT/"SkeletonData.zip", ROOT/"simplified_partial.zip"] if p.exists())
OUT = ROOT/"resultados"; OUT.mkdir(exist_ok=True)
SEEDS = [0, 1, 2]; T_FIX = 64; GESTO = 2; EXCLUIR_SUJETO = "217"
torch.set_num_threads(4)
print("fuente:", ZIP.name, "| python", platform.python_version(), "| numpy", np.__version__,
      "| sklearn", sklearn.__version__, "| torch", torch.__version__)


## 1. Carga y flujo de exclusiones
El nombre del archivo es `Sujeto_Fecha_Gesto_Repetición_Etiqueta_Postura.txt`. El identificador del sujeto
se usa SOLO para agrupar las particiones; nunca entra al modelo. Tampoco entran grupo ni postura.

In [ ]:
recs = []
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if "/Simplified/" not in n or not n.endswith(".txt"): continue
        p = pathlib.PurePosixPath(n).stem.split("_")
        if int(p[2]) != GESTO: continue
        a = np.loadtxt(io.StringIO(z.read(n).decode("utf-8-sig")), delimiter=",", ndmin=2).reshape(-1, 25, 3)
        recs.append(dict(file=n, subj=p[0], label=int(p[4]), pos="_".join(p[5:]), xyz=a.astype(np.float32)))
flujo = [("gesto 2, todo", len(recs), len({r["subj"] for r in recs}))]
recs = [r for r in recs if r["subj"] != EXCLUIR_SUJETO]
flujo.append(("− grabación interrumpida (aviso de Zenodo)", len(recs), len({r["subj"] for r in recs})))
recs = [r for r in recs if r["label"] in (1, 2)]
flujo.append(("− etiqueta 3 (ambigua)", len(recs), len({r["subj"] for r in recs})))
recs.sort(key=lambda r: r["file"])
for f in flujo: print(f"{f[0]:<45} secuencias={f[1]:>4}  participantes={f[2]:>3}")
y = np.array([r["label"] == 2 for r in recs], int)          # 1 = "incorrecta" (clase de interés)
groups = np.array([r["subj"] for r in recs])
es_paciente = np.array([s.startswith("2") for s in groups])  # 2xx = pacientes (artículo); SOLO para auditar
print("clases:", collections.Counter(y), "| incorrectas hechas por pacientes:", int(y[es_paciente].sum()), "de", int(y.sum()))
print("participantes con alguna incorrecta:", len({g for g, t in zip(groups, y) if t}),
      "| con alguna correcta:", len({g for g, t in zip(groups, y) if not t}))
top3 = collections.Counter(groups[y == 1]).most_common(3)
print("las 3 personas con más incorrectas aportan", sum(c for _, c in top3), "de", int(y.sum()))

## 2. Entradas
- **Normalización por secuencia (no aprende nada de otros datos):** cada fotograma se centra en la base de la
  columna y se divide por la longitud del tronco de esa persona → reduce diferencias de posición y escala corporal, sin eliminar todas las diferencias entre personas.
- **Red:** la secuencia se remuestrea a 64 fotogramas (75 canales). Como eso borra la duración, se añade
  log(número de fotogramas) como dato aparte. Es un indicador aproximado de duración, no tiempo real medido por timestamps.
- **Comparador:** 15 características resumidas de ángulos (ambos brazos, para no depender de la convención de lado).

In [ ]:
def normaliza(a):
    a = a - a[:, :1, :]
    torso = np.median(np.linalg.norm(a[:, 20], axis=1)) + 1e-6
    return a / torso
def remuestrea(a, T=T_FIX):
    t0, t1 = np.linspace(0, 1, len(a)), np.linspace(0, 1, T)
    flat = a.reshape(len(a), -1)
    return np.stack([np.interp(t1, t0, flat[:, k]) for k in range(flat.shape[1])], 0).astype(np.float32)  # (75,T)
def ang(u, v):
    c = (u * v).sum(-1) / (np.linalg.norm(u, axis=-1) * np.linalg.norm(v, axis=-1) + 1e-9)
    return np.degrees(np.arccos(np.clip(c, -1, 1)))
def caracteristicas(a):
    f = []
    tronco = a[:, 0] - a[:, 20]
    for sh, el, wr in [(8, 9, 10), (4, 5, 6)]:
        elev = ang(a[:, el] - a[:, sh], tronco); codo = ang(a[:, sh] - a[:, el], a[:, wr] - a[:, el])
        f += [elev.max(), elev.min(), np.ptp(elev), codo.min(), codo.mean(), np.abs(np.diff(elev, 2)).mean() if len(a) > 2 else 0]
    incl = ang(a[:, 20] - a[:, 0], np.array([[0, 1, 0]], np.float32))
    f += [incl.max(), np.ptp(incl), np.log(len(a))]
    return f
NORM = [normaliza(r["xyz"]) for r in recs]
X_seq = np.stack([remuestrea(a) for a in NORM])                      # (N,75,64)
logdur = np.log([len(r["xyz"]) for r in recs]).astype(np.float32)
X_feat = np.array([caracteristicas(r["xyz"]) for r in recs], np.float32)
X_static = np.stack([a[0].reshape(-1) for a in NORM])                # solo el PRIMER fotograma (sin movimiento)
print(X_seq.shape, X_feat.shape, X_static.shape)

## 3. Modelos
**CNN temporal pequeña:** 2 capas convolucionales 1D (filtros que recorren el tiempo), promedio global, dropout
y una salida. 22.594 parámetros entrenables. Su tamaño no garantiza que sea adecuado para la diversidad disponible; debe evaluarse el sobreajuste. Pérdida: entropía cruzada binaria con peso mayor
para la clase minoritaria (calculado solo con entrenamiento). Optimizador Adam.

### Qué ocurre al entrenar
Una época es una pasada por los ejemplos de entrenamiento. Cada lote contiene hasta 32 secuencias: la red predice, la pérdida compara con las etiquetas, la retropropagación calcula gradientes y Adam actualiza los pesos. Tasa de aprendizaje: 0,001; weight decay: 0,0001; dropout: 0,30. Las clases se ponderan a partir del entrenamiento.

Entrada: 75 coordenadas por 64 instantes. Primera convolución: 32 filtros de tamaño 5; segunda: 64 filtros de tamaño 5. BatchNorm y ReLU siguen a ambas; MaxPool reduce el tiempo tras la primera y el promedio global resume tras la segunda. Se añade el indicador de duración antes de la salida. Sigmoide transforma la salida en una puntuación; no se ha demostrado calibración clínica.


In [ ]:
class CNNTemporal(nn.Module):
    def __init__(s, c=75):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv1d(c, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
                               nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        s.drop = nn.Dropout(0.3); s.out = nn.Linear(65, 1)
    def forward(s, x, d):
        return s.out(torch.cat([s.drop(s.conv(x).squeeze(-1)), d[:, None]], 1)).squeeze(-1)
print("parámetros CNN:", sum(p.numel() for p in CNNTemporal().parameters()))
MAX_EP = 60
def prepara(Xtr, dtr, *otros):
    m, s = Xtr.mean((0, 2), keepdims=True), Xtr.std((0, 2), keepdims=True) + 1e-6   # estadísticas SOLO de entrenamiento
    dm, ds = dtr.mean(), dtr.std() + 1e-6
    return [(torch.tensor((X - m) / s), torch.tensor((d - dm) / ds)) for X, d in [(Xtr, dtr), *otros]]
def entrena(Xtr, dtr, ytr, epocas, seed, val=None, estados=None):
    torch.manual_seed(seed); np.random.seed(seed)
    tensores = prepara(Xtr, dtr, *( [val[:2]] if val else [] ))
    xt, dt = (t.float() for t in tensores[0]); yt = torch.tensor(ytr, dtype=torch.float32)
    if val: xv, dv = (t.float() for t in tensores[1]); yv = torch.tensor(val[2], dtype=torch.float32)
    net = CNNTemporal(); opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((ytr == 0).sum() / max(1, (ytr == 1).sum()), dtype=torch.float32))
    hist = []
    for ep in range(epocas):
        net.train(); perm = torch.randperm(len(xt)); tot = 0.0
        for i in range(0, len(xt), 32):
            b = perm[i:i + 32]
            if len(b) < 2: continue
            opt.zero_grad(); l = lossf(net(xt[b], dt[b]), yt[b]); l.backward(); opt.step(); tot += l.item() * len(b)
        if val:
            net.eval()
            with torch.no_grad():
                lv = net(xv, dv); vl = nn.functional.binary_cross_entropy_with_logits(lv, yv).item()
            hist.append((tot / len(xt), vl, torch.sigmoid(lv).numpy()))
            if estados is not None: estados.append({k: v.detach().clone() for k, v in net.state_dict().items()})
    return net, tensores, hist
def predice(net, Xtr, dtr, X, d):
    t = prepara(Xtr, dtr, (X, d))[1]; net.eval()
    with torch.no_grad(): return torch.sigmoid(net(*t)).numpy()
def umbral_youden(y_true, p):
    cand = np.unique(np.round(p, 3)); best = (-1, 0.5)
    for t in cand:
        pr = p >= t; se = pr[y_true == 1].mean(); sp = (~pr[y_true == 0]).mean()
        if se + sp > best[0]: best = (se + sp, t)
    return float(best[1])


## 4. Validación cruzada ANIDADA y agrupada por participante
Solo 10 personas tienen secuencias «incorrectas»: un único test reservado tendría 2–3 de ellas y sería inestable.
Por eso: **bucle externo** (5 particiones por participante) = estima desempeño; **bucle interno** (3 particiones,
solo con el entrenamiento externo) = elige hiperparámetro (C de la logística / nº de épocas de la CNN) y el
**umbral** (índice de Youden). El fold externo de prueba no participa en ninguna decisión. Se repite con 3 semillas.

### Cómo elegimos las épocas
El bucle interno recorre 60 epocas y registra la perdida de entrenamiento y validacion. Se selecciona la epoca de menor perdida media de validacion interna. Despues se reutilizan los estados ya entrenados de esa epoca y se promedia la prediccion de las tres particiones internas, sin iniciar otro entrenamiento. No es una detencion anticipada del bucle de 60 epocas. La prueba externa no elige epocas ni umbral.

En A se usan 5 particiones externas; en B, 4. Ambas usan 3 internas y semillas 0, 1 y 2. Todas las repeticiones de una persona permanecen juntas. Las semillas no son nuevos participantes ni estudios independientes.


In [ ]:
def metricas(y_true, pred, p=None):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = tn + fp + fn + tp
    se = tp / (tp + fn) if tp + fn else np.nan
    sp = tn / (tn + fp) if tn + fp else np.nan
    pr = tp / (tp + fp) if tp + fp else np.nan
    m = dict(sens=se, espec=sp, prec=pr, f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
             bal_acc=(se + sp) / 2, acc=(tp + tn) / max(1, n), errores=int(fp + fn),
             mcc=matthews_corrcoef(y_true, pred), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    if p is not None:
        pp = np.clip(p, 1e-7, 1 - 1e-7)
        m["brier"] = brier_score_loss(y_true, pp)
        m["logloss"] = log_loss(y_true, pp, labels=[0, 1])
        if len(set(y_true)) == 2:
            m["auc"] = roc_auc_score(y_true, p)
            m["pr_auc"] = average_precision_score(y_true, p)
    return m
def lr_anidada(X, ytr, gtr, Xte, seed):
    inner = StratifiedGroupKFold(3, shuffle=True, random_state=seed); best = None
    for C in [0.01, 0.1, 1, 10]:
        oof = np.zeros(len(ytr))
        for a, b in inner.split(X, ytr, gtr):
            oof[b] = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=2000)).fit(X[a], ytr[a]).predict_proba(X[b])[:, 1]
        auc = roc_auc_score(ytr, oof)
        if best is None or auc > best[0]: best = (auc, C, oof.copy())
    thr = umbral_youden(ytr, best[2])
    mdl = make_pipeline(StandardScaler(), LogisticRegression(C=best[1], class_weight="balanced", max_iter=2000)).fit(X, ytr)
    return mdl.predict_proba(Xte)[:, 1], thr, dict(C=best[1])
def cnn_anidada(Xs, d, ytr, gtr, Xte, dte, seed, curvas=None):
    # Una trayectoria por fold interno: se conserva el estado de cada epoca y se reutiliza el seleccionado.
    inner = StratifiedGroupKFold(3, shuffle=True, random_state=seed)
    vls, trl, estados = [], [], []; probs = np.zeros((MAX_EP, len(ytr)))
    for a, b in inner.split(Xs, ytr, gtr):
        estados_fold = []
        _, _, h = entrena(Xs[a], d[a], ytr[a], MAX_EP, seed, val=(Xs[b], d[b], ytr[b]), estados=estados_fold)
        vls.append([e[1] for e in h]); trl.append([e[0] for e in h]); estados.append(estados_fold)
        for ep, e in enumerate(h): probs[ep, b] = e[2]
    vl = np.mean(vls, 0); ep = int(np.argmin(vl)) + 1
    if curvas is not None: curvas.append((np.mean(trl, 0), vl, ep))
    thr = umbral_youden(ytr, probs[ep - 1])
    net = CNNTemporal(); partes = []
    for estado, (a, b) in zip(estados, inner.split(Xs, ytr, gtr)):
        net.load_state_dict(estado[ep - 1]); net.eval()
        partes.append(predice(net, Xs[a], d[a], Xte, dte))  # misma normalizacion que al entrenar este modelo interno
    return np.mean(partes, 0), thr, dict(epocas=ep)

def corre(idx, n_outer, nombre):
    """idx: índices del subconjunto analizado. Devuelve predicciones fuera de muestra por semilla y modelo."""
    res = collections.defaultdict(list); curvas = []; particiones = []
    for seed in SEEDS:
        outer = StratifiedGroupKFold(n_outer, shuffle=True, random_state=seed)
        P = {m: np.full(len(idx), np.nan) for m in ["estatico", "logistica", "cnn"]}; H = {m: np.zeros(len(idx), int) for m in P}
        for k, (a, b) in enumerate(outer.split(idx, y[idx], groups[idx])):
            A, B = idx[a], idx[b]
            assert not set(groups[A]) & set(groups[B])                     # ninguna persona en ambos lados
            particiones.append(dict(analisis=nombre, seed=seed, fold=k, test_subjects=sorted(set(groups[B])),
                                    n_train=len(A), n_test=len(B), test_incorrectas=int(y[B].sum())))
            for m, (p, thr, hp) in {"estatico": lr_anidada(X_static[A], y[A], groups[A], X_static[B], seed),
                                    "logistica": lr_anidada(X_feat[A], y[A], groups[A], X_feat[B], seed),
                                    "cnn": cnn_anidada(X_seq[A], logdur[A], y[A], groups[A], X_seq[B], logdur[B], seed, curvas if seed == 0 else None)}.items():
                P[m][b] = p; H[m][b] = (p >= thr)
                res[m + "_folds"].append(dict(seed=seed, fold=k, umbral=thr, **hp, **metricas(y[B], p >= thr, p)))
        for m in P: res[m + "_pooled"].append(dict(seed=seed, **metricas(y[idx], H[m], P[m]))); res[m + "_pred"].append(H[m]); res[m + "_prob"].append(P[m])
    return res, curvas, particiones

t0 = time.time()
todos = np.arange(len(y)); solo_pac = np.where(es_paciente)[0]
R1, C1, PT1 = corre(todos, 5, "A_todos")
print("análisis A listo", round(time.time() - t0), "s")
R2, C2, PT2 = corre(solo_pac, 4, "B_solo_pacientes")
print("análisis B listo", round(time.time() - t0), "s")


## 5. Resultados
**A = todos los participantes** **B = solo pacientes**: control de sesgo, porque
en el gesto 2 *todas* las incorrectas son de pacientes y *todos* los controles son 100 % correctos; un modelo
podría «acertar» reconociendo quién es paciente (postura, silla de ruedas) sin mirar la calidad del movimiento.
El modelo **estático** (solo el primer fotograma, sin movimiento) explora la capacidad predictiva de la postura inicial; no identifica causalmente el atajo usado por la CNN. La normalización del control estático original utiliza la secuencia completa, aspecto revisado mediante un control estricto posterior.

In [ ]:
def resumen(R, titulo):
    print(f"\n=== {titulo} ===")
    print(f"{'modelo':<10} {'sens':>12} {'espec':>12} {'prec':>12} {'F1':>12} {'bal_acc':>12} {'AUC':>12}   (media ± DE entre semillas, predicciones agregadas)")
    out = {}
    for m in ["estatico", "logistica", "cnn"]:
        pool = R[m + "_pooled"]; row = {k: (float(np.mean([q[k] for q in pool])), float(np.std([q[k] for q in pool]))) for k in ["sens", "espec", "prec", "f1", "bal_acc", "auc"]}
        folds = R[m + "_folds"]; row["bal_acc_entre_folds"] = (float(np.nanmean([q["bal_acc"] for q in folds])), float(np.nanstd([q["bal_acc"] for q in folds])))
        row["folds_bal_acc_validos"] = sum(bool(np.isfinite(q["bal_acc"])) for q in folds)
        row["folds_totales"] = len(folds)
        out[m] = row
        print(f"{m:<10} " + " ".join(f"{row[k][0]:.2f} ± {row[k][1]:.2f}".rjust(12) for k in ["sens", "espec", "prec", "f1", "bal_acc", "auc"]),
              f"  | bal_acc por fold: {row['bal_acc_entre_folds'][0]:.2f} ± {row['bal_acc_entre_folds'][1]:.2f}")
    for m in ["logistica", "cnn"]:
        q = R[m + "_pooled"][0]; print(f"matriz de confusión {m} (semilla 0): VN={q['tn']} FP={q['fp']} FN={q['fn']} VP={q['tp']}")
    return out
S1 = resumen(R1, "A · todos los participantes (369 secuencias, 26 personas)")
S2 = resumen(R2, f"B · solo pacientes ({len(solo_pac)} secuencias, {len(set(groups[solo_pac]))} personas)")

### Metricas complementarias de clasificacion y generalizacion
Accuracy y cantidad absoluta de errores siguen el criterio usado en clase. PR-AUC evalua la clase minoritaria incorrecta bajo desbalance; MCC resume la matriz completa. Brier y log-loss miden la calidad de las probabilidades. La comparacion CNN menos logistica usa las mismas predicciones fuera de muestra. La brecha de perdida usa los minimos internos ya elegidos para seleccionar epocas: un valor positivo indica que la validacion quedo por encima del entrenamiento.


In [ ]:
CLAVES_EXT = ["acc", "errores", "mcc", "pr_auc", "brier", "logloss"]

def resumir_extension(R, titulo):
    print(f"\n=== {titulo}: metricas complementarias ===")
    encabezado = f"{'modelo':<10}" + "".join(f"{k:>14}" for k in CLAVES_EXT)
    print(encabezado + "   (media +/- DE entre semillas)")
    out = {}
    for m in ["estatico", "logistica", "cnn"]:
        pool = R[m + "_pooled"]
        row = {k: (float(np.mean([q[k] for q in pool])), float(np.std([q[k] for q in pool]))) for k in CLAVES_EXT}
        out[m] = row
        print(f"{m:<10}" + "".join(
            f"{row[k][0]:>8.0f} +/- {row[k][1]:<4.0f}" if k == "errores" else f"{row[k][0]:>8.2f} +/- {row[k][1]:<4.2f}"
            for k in CLAVES_EXT))
    print("diferencia CNN menos logistica:")
    for k in ["acc", "mcc", "pr_auc", "auc", "brier", "logloss"]:
        d = [R["cnn_pooled"][i][k] - R["logistica_pooled"][i][k] for i in range(len(SEEDS))]
        print(f"  {k:<8} {np.mean(d):+.3f} +/- {np.std(d):.3f}")
    return out

S1_ext = resumir_extension(R1, "A - todos")
S2_ext = resumir_extension(R2, "B - solo pacientes")
print("Las perdidas de entrenamiento y validacion tienen ponderaciones diferentes; no se calcula su diferencia como brecha de generalizacion.")


### Incertidumbre: bootstrap por participante (semilla 0)
Se remuestrean PERSONAS, no secuencias, porque las repeticiones de una persona no son independientes.

In [ ]:
def boot(idx, pred, n=2000, seed=0):
    rng = np.random.default_rng(seed); sub = sorted(set(groups[idx])); pos = {s: np.where(groups[idx] == s)[0] for s in sub}; vals = []
    for _ in range(n):
        ii = np.concatenate([pos[s] for s in rng.choice(sub, len(sub))]); yt = y[idx][ii]
        if len(set(yt)) < 2: continue
        m = metricas(yt, pred[ii]); vals.append([m["sens"], m["espec"], m["bal_acc"]])
    return np.percentile(vals, [2.5, 97.5], 0).T
IC = {}
for nom, idx, R in [("A", todos, R1), ("B", solo_pac, R2)]:
    for m in ["estatico", "logistica", "cnn"]:
        ci = boot(idx, R[m + "_pred"][0]); IC[f"{nom}_{m}"] = ci.tolist()
        print(f"{nom} {m:<10} IC95% sens [{ci[0][0]:.2f}, {ci[0][1]:.2f}]  espec [{ci[1][0]:.2f}, {ci[1][1]:.2f}]  bal_acc [{ci[2][0]:.2f}, {ci[2][1]:.2f}]")

### Desempeño por participante y fallos (CNN, análisis A, semilla 0)

In [ ]:
pred = R1["cnn_pred"][0]; por_persona = []
for s in sorted(set(groups)):
    i = groups == s
    por_persona.append(dict(sujeto=s, grupo="paciente" if s.startswith("2") else "control", n=int(i.sum()), incorrectas=int(y[i].sum()),
                            aciertos=int((pred[i] == y[i]).sum()), fp=int(((pred == 1) & (y == 0) & i).sum()), fn=int(((pred == 0) & (y == 1) & i).sum())))
for q in por_persona:
    if q["fp"] + q["fn"]: print(q)
print("controles con algún falso positivo:", sum(1 for q in por_persona if q["grupo"] == "control" and q["fp"]))
pc = (y == 0) & es_paciente
print("especificidad SOLO en secuencias correctas de pacientes (análisis A): "
      f"logística {np.mean(R1['logistica_pred'][0][pc] == 0):.2f} | cnn {np.mean(pred[pc] == 0):.2f}  (n={int(pc.sum())})")

## Epocas de esta ejecucion

Consultar la salida dinamica debajo de las curvas. Se retira la tabla historica para no mezclar ejecuciones. Azul: perdida ponderada de entrenamiento; naranja: perdida no ponderada de validacion interna. No interpretar su resta como medida de generalizacion. Las lineas verticales indican la epoca seleccionada mediante validacion interna.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, (C, t) in enumerate([(C1, "A · todos"), (C2, "B · solo pacientes")]):
    for tr, vl, ep in C: ax[k].plot(tr, color="#2b6cb0", alpha=.35); ax[k].plot(vl, color="#d9480f", alpha=.35); ax[k].axvline(ep - 1, color="#999", lw=.5)
    ax[k].plot([], [], color="#2b6cb0", label="pérdida entrenamiento"); ax[k].plot([], [], color="#d9480f", label="pérdida validación interna")
    ax[k].set_title(f"CNN, curvas por fold externo (semilla 0) — {t}"); ax[k].set_xlabel("época"); ax[k].legend()
plt.tight_layout(); plt.savefig(OUT/"curvas_cnn.png", dpi=130); print("guardado", OUT/"curvas_cnn.png")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / "curvas_cnn.png")))
print("Figura de esta ejecucion: x=0 corresponde a época 1.")
print("Épocas seleccionadas en ESTA ejecución (semilla 0):")
for nombre, curvas in [("A: todos", C1), ("B: pacientes", C2)]:
    print(nombre, [int(curva[2]) for curva in curvas])


In [ ]:
pesos_ref = hashlib.sha256(X_seq.tobytes()).hexdigest()[:16]
payload = dict(fecha=time.strftime("%Y-%m-%d"), fuente=ZIP.name, entorno=dict(python=platform.python_version(), numpy=np.__version__, sklearn=sklearn.__version__, torch=torch.__version__),
               semillas=SEEDS, T_FIX=T_FIX, flujo_exclusiones=flujo, huella_entradas=pesos_ref,
               A_todos=S1, B_solo_pacientes=S2, A_complementarias=S1_ext, B_complementarias=S2_ext,
               IC95_bootstrap_por_participante_semilla0=IC, por_participante_cnn_A=por_persona,
               folds={k: v for R, pre in [(R1, "A_"), (R2, "B_")] for k, v in ((pre + kk, vv) for kk, vv in R.items() if kk.endswith("_folds"))})

def limpio(v):
    if isinstance(v, dict): return {k: limpio(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [limpio(x) for x in v]
    if isinstance(v, np.ndarray): return limpio(v.tolist())
    if isinstance(v, (float, np.floating)): return float(v) if np.isfinite(v) else None
    if isinstance(v, np.integer): return int(v)
    return v

payload["revision"] = "2026-10-05_normalizacion_interna_metricas_clase_ausente"
payload["entorno"]["matplotlib"] = matplotlib.__version__
json.dump(limpio(payload), open(OUT/"resultados.json", "w"), ensure_ascii=False, indent=1, allow_nan=False)
json.dump(PT1 + PT2, open(OUT/"particiones.json", "w"), indent=1)
torch.manual_seed(0); net, _, _ = entrena(X_seq, logdur, y, int(np.median([c[2] for c in C1])), 0)
torch.save(net.state_dict(), OUT/"cnn_demo_todos_los_datos.pt")   # SOLO para demo; no tiene evaluación independiente
print("listo en", round(time.time() - t0), "s")

# Exportar predicciones para auditoria sin reentrenar.
predicciones = {}
for nombre, idx, R in [("A", todos, R1), ("B", solo_pac, R2)]:
    predicciones[nombre] = {"indices": idx.tolist(), "y": y[idx].tolist(), "grupos": groups[idx].tolist(), "modelos": {m: {"pred": [v.tolist() for v in R[m + "_pred"]], "prob": [v.tolist() for v in R[m + "_prob"]]} for m in ["estatico", "logistica", "cnn"]}}
json.dump(limpio(predicciones), open(OUT/"predicciones_auditoria.json", "w"), allow_nan=False)


## Interpretacion pendiente de la ejecucion corregida

No reutilizar las cifras de versiones anteriores. Redactar las conclusiones a partir de las nuevas salidas. Las particiones con una sola clase no permiten estimar sensibilidad/especificidad de la clase ausente ni exactitud balanceada binaria; se exportan como null y se excluyen del promedio por fold, informando su cantidad. Las metricas agrupadas por semilla se calculan sobre todas sus predicciones externas.

La CNN promedia tres modelos internos con sus propias normalizaciones; no se reentrena sobre todo el entrenamiento externo. El umbral procede de predicciones internas fuera de muestra, mientras la prueba usa el promedio de modelos: documentar esta diferencia como limitacion. Las semillas no son estudios independientes. Sin validacion clinica.


## Descargar resultados de esta ejecución

Este ZIP contiene resultados.json, particiones, curvas y pesos de demostración. No incluye el dataset.


In [ ]:
import shutil
ruta_zip = shutil.make_archive(str(ROOT / "resultados_TP_IRDS"), "zip", root_dir=OUT)
files.download(ruta_zip)
